# Aula 02 — KDD passos 1 a 3 (ENEM 2025 RN) — versão Colab

Consolida os 3 notebooks do AVA (`aula2_passo1_selecao`, `aula2_passo2_preprocessamento`, `aula2_passo3_transformacao`) com as correções listadas no `README.md` (marcadas com `# [FIX]`).

**Pergunta de pesquisa:** Conseguimos prever o nível de desempenho de um aluno do ENEM 2025 (RN) com base nas notas das provas objetivas e no perfil da escola?

**Arquivos (baixar do AVA):** `RESULTADOS_2025_RN.csv`, `catalogo_escolas_rn.csv`.

In [ ]:
# === SETUP COLAB ===
# Alternativa com Drive: from google.colab import drive; drive.mount("/content/drive"); os.chdir("/content/drive/MyDrive/<pasta>")
import os, shutil
try:
    from google.colab import files
    enviados = files.upload()
    for nome in enviados:
        if "RESULTADO" in nome.upper() or "MICRODADOS" in nome.upper():
            shutil.move(nome, "RESULTADOS_2025_RN.csv")
        elif "CATALOGO" in nome.upper() or "CATÁLOGO" in nome.upper():
            shutil.move(nome, "catalogo_escolas_rn.csv")
except ImportError:
    pass
print([f for f in os.listdir(".") if f.endswith(".csv")])

## Passo 1 — Seleção

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

RESULTADOS = "RESULTADOS_2025_RN.csv"
CATALOGO   = "catalogo_escolas_rn.csv"

print("=" * 70)
print("PASSO 1 — SELEÇÃO")
print("=" * 70)

df_raw = pd.read_csv(RESULTADOS, sep=";", encoding="latin1")
df_cat = pd.read_csv(CATALOGO,   sep=",", encoding="utf-8")   # [FIX] original usava latin1

print(f"\n[RAW] Registros totais no arquivo de resultados   : {len(df_raw):,}")
print(f"[RAW] Colunas disponíveis                        : {df_raw.shape[1]}")
print(f"[CAT] Escolas no catálogo                        : {len(df_cat):,}")
print("[CAT] Colunas do catálogo:", list(df_cat.columns))

In [ ]:
presenca_completa = (
    (df_raw["TP_PRESENCA_CN"] == 1) &
    (df_raw["TP_PRESENCA_CH"] == 1) &
    (df_raw["TP_PRESENCA_LC"] == 1) &
    (df_raw["TP_PRESENCA_MT"] == 1)
)

df_presentes = df_raw[presenca_completa].copy()

print(f"\n [SELEÇÃO] Candidatos com presença completa      : {len(df_presentes):,}")   # esperado: 25.965
print(f"[EXCLUSÃO] Descartados por ausência              : {len(df_raw) - len(df_presentes):,}")

In [ ]:
COLUNAS = [
    "CO_ESCOLA",               # chave para join com catálogo
    "NO_MUNICIPIO_ESC",        # município da escola
    "TP_DEPENDENCIA_ADM_ESC",  # 1=Federal, 2=Estadual, 3=Municipal, 4=Privada
    "TP_LOCALIZACAO_ESC",      # 1=Urbana, 2=Rural
    "NU_NOTA_CN",              # Ciências da Natureza
    "NU_NOTA_CH",              # Ciências Humanas
    "NU_NOTA_LC",              # Linguagens e Códigos
    "NU_NOTA_MT",              # Matemática
    "NU_NOTA_REDACAO",         # Redação
]

df_sel = df_presentes[COLUNAS].copy()
df_sel = df_sel.merge(df_cat, on="CO_ESCOLA", how="left")

# [FIX] TIPO_ESCOLA não existe no dicionário de dados; criar a partir da dependência administrativa
if "TIPO_ESCOLA" not in df_sel.columns:
    df_sel["TIPO_ESCOLA"] = df_sel["TP_DEPENDENCIA_ADM_ESC"].map(
        {1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada"})

print(f"\n[SELEÇÃO] Colunas selecionadas                  : {df_sel.shape[1]}")
print(f"[SELEÇÃO] Escolas identificadas pelo catálogo    : "
      f"{df_sel['NO_ESCOLA'].notna().sum():,} / {len(df_sel):,}")
print("\n[SELEÇÃO] Candidatos por tipo de escola:")
print(df_sel["TIPO_ESCOLA"].value_counts(dropna=False).to_string())
df_sel.head()

In [ ]:
df_sel.to_csv("dados_selecionados.csv", index=False, encoding="utf-8")
print(f"\n[OK] Arquivo salvo: dados_selecionados.csv  ({len(df_sel):,} registros, {df_sel.shape[1]} colunas)")

## Passo 2 — Pré-processamento

In [ ]:
df = pd.read_csv("dados_selecionados.csv", encoding="utf-8")

print("=" * 65)
print("PASSO 2 — PRÉ-PROCESSAMENTO")
print("=" * 65)
print(f"\n[ENTRADA] {len(df):,} registros, {df.shape[1]} colunas")

NOTAS = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO"]

print("\n[DIAGNÓSTICO] Valores ausentes (NaN) por coluna:")
for col in NOTAS:
    n = df[col].isna().sum()
    print(f"  {col} : {n} ({n/len(df)*100:.1f}%)")

print("\n[DIAGNÓSTICO] Notas zero (suspeitas) por coluna:")
for col in NOTAS:
    n = (df[col] == 0).sum()
    print(f"  {col}: {n} ({n/len(df)*100:.1f}%)")

In [ ]:
antes = len(df)
for col in NOTAS:
    df[col] = df[col].replace(0, np.nan)
print(f"\n[TRATAMENTO] Notas zero convertidas para NaN em: {NOTAS}")

df_clean = df.dropna(subset=NOTAS).copy()
removidos = antes - len(df_clean)
print(f"[TRATAMENTO] Registros removidos por nota NaN    : {removidos:,}")
print(f"[RESULTADO]  Registros restantes                 : {len(df_clean):,}")

dupl = df_clean.duplicated().sum()
print(f"\n[DIAGNÓSTICO] Registros duplicados              : {dupl}")
if dupl:
    df_clean = df_clean.drop_duplicates()
    print("[TRATAMENTO]  Duplicatas removidas.")

In [ ]:
# 2.5 Checar e corrigir tipos de dados
df_clean["CO_ESCOLA"] = df_clean["CO_ESCOLA"].astype("Int64").astype(str)   # [FIX] evita "24012345.0"
df_clean["TP_LOCALIZACAO_ESC"] = df_clean["TP_LOCALIZACAO_ESC"].astype("Int64")
df_clean["TP_DEPENDENCIA_ADM_ESC"] = df_clean["TP_DEPENDENCIA_ADM_ESC"].astype("Int64")

# 2.6 Padronizar escola sem match no catálogo
sem_nome = df_clean["NO_ESCOLA"].isna().sum()
if sem_nome:
    df_clean["NO_ESCOLA"] = df_clean["NO_ESCOLA"].fillna("Escola não identificada")
    print(f"\n[TRATAMENTO] Escolas sem match no catálogo      : {sem_nome} → preenchidas")
df_clean.dtypes

In [ ]:
# 2.7 Boxplots após a limpeza
LABELS = ["Ciências\nNatureza", "Ciências\nHumanas", "Linguagens", "Matemática", "Redação"]
CORES  = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#76b7b2"]

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.suptitle("Distribuição das Notas após Pré-processamento\n(candidatos com presença completa e notas válidas)",
             fontsize=12, fontweight="bold")

for ax, col, label, cor in zip(axes, NOTAS, LABELS, CORES):
    ax.boxplot(df_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor=cor, alpha=0.7),
               medianprops=dict(color="black", linewidth=2))
    ax.set_title(label, fontsize=10)
    ax.set_ylabel("Nota (0–1000)")
    ax.set_xticks([])
    media = df_clean[col].mean()
    mediana = df_clean[col].median()
    ax.axhline(media, color="navy", linestyle="--", linewidth=1, label=f"Média: {media:.0f}")
    ax.axhline(mediana, color="black", linestyle="-", linewidth=1, label=f"Mediana: {mediana:.0f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("passo2_boxplots.png", dpi=120, bbox_inches="tight")   # [FIX] salvar ANTES do show
plt.show()

In [ ]:
# 2.8 Relatório final
print("\n[RESUMO PRÉ-PROCESSAMENTO]")
print(f"  Registros de entrada  : {antes:,}")
print(f"  Registros de saída    : {len(df_clean):,}")
print(f"  Perda total           : {antes - len(df_clean):,} ({(antes-len(df_clean))/antes*100:.1f}%)   # meta: < 15%")
print("\n[ESTATÍSTICAS FINAIS DAS NOTAS]")
print(df_clean[NOTAS].describe().round(2).to_string())

df_clean.to_csv("dados_preprocessados.csv", index=False, encoding="utf-8")
print(f"\n[OK] Arquivo salvo: dados_preprocessados.csv  ({len(df_clean):,} registros)")

## Passo 3 — Transformação

In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

df = pd.read_csv("dados_preprocessados.csv", encoding="utf-8")
print("=" * 65)
print("PASSO 3 — TRANSFORMAÇÃO")
print("=" * 65)
print(f"\n[ENTRADA] {len(df):,} registros")

NOTAS_OBJ   = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT"]
NOTAS_TODAS = NOTAS_OBJ + ["NU_NOTA_REDACAO"]

# 3.1 Atributos derivados numéricos
df["MEDIA_OBJETIVAS"] = df[NOTAS_OBJ].mean(axis=1).round(2)
df["MEDIA_GERAL"] = ((df[NOTAS_OBJ].sum(axis=1) + df["NU_NOTA_REDACAO"]) / 5).round(2)
df["DESVIO_NOTAS"] = df[NOTAS_TODAS].std(axis=1).round(2)
print("\n[FEATURE] MEDIA_OBJETIVAS, MEDIA_GERAL, DESVIO_NOTAS criados")

In [ ]:
# 3.2 Atributos derivados categóricos
MAPA_AREA = {
    "NU_NOTA_CN": "Ciências Natureza",
    "NU_NOTA_CH": "Ciências Humanas",
    "NU_NOTA_LC": "Linguagens",
    "NU_NOTA_MT": "Matemática",
    "NU_NOTA_REDACAO": "Redação",
}
df["MELHOR_AREA"] = df[NOTAS_TODAS].idxmax(axis=1).map(MAPA_AREA)
df["PIOR_AREA"]   = df[NOTAS_TODAS].idxmin(axis=1).map(MAPA_AREA)

print("[FEATURE] MELHOR_AREA e PIOR_AREA criados")
print("\n  Melhor área mais comum:")
print(df["MELHOR_AREA"].value_counts().to_string())
print("\n  Pior área mais comum:")
print(df["PIOR_AREA"].value_counts().to_string())

In [ ]:
# 3.3 Codificação de variáveis categóricas
df["LOCALIZACAO_RURAL"] = (df["TP_LOCALIZACAO_ESC"] == 2).astype(int)          # binária

df["TIPO_ESCOLA"] = df["TIPO_ESCOLA"].fillna("Estadual")
dummies = pd.get_dummies(df["TIPO_ESCOLA"], prefix="ESCOLA", drop_first=False)  # One-Hot
df = pd.concat([df, dummies], axis=1)

le = LabelEncoder()                                                             # Label (tree-based)
df["MELHOR_AREA_COD"] = le.fit_transform(df["MELHOR_AREA"])
df["PIOR_AREA_COD"]   = le.fit_transform(df["PIOR_AREA"])
print("\n[ENCODING] LOCALIZACAO_RURAL, One-Hot TIPO_ESCOLA, Label MELHOR/PIOR_AREA")

In [ ]:
# 3.4 Variável-alvo
LIMITES = [0, 400, 550, 700, 1001]
ROTULOS = ["Insuficiente", "Regular", "Bom", "Excelente"]

df["NIVEL_DESEMPENHO"] = pd.cut(df["MEDIA_GERAL"], bins=LIMITES, labels=ROTULOS, right=False)

print("\n[TARGET] Distribuição de NIVEL_DESEMPENHO:   (ref.: ~8% / ~38% / ~41% / ~13%)")
contagem = df["NIVEL_DESEMPENHO"].value_counts().sort_index()
for nivel, n in contagem.items():
    print(f"  {nivel:<14}: {n:>5} ({n/len(df)*100:.1f}%)")

In [ ]:
# 3.5 Normalização Min-Max (KNN, SVM, redes neurais)
scaler = MinMaxScaler()
COLUNAS_NORM = NOTAS_TODAS + ["MEDIA_OBJETIVAS", "MEDIA_GERAL", "DESVIO_NOTAS"]
df_norm = df.copy()
df_norm[[c + "_NORM" for c in COLUNAS_NORM]] = scaler.fit_transform(df[COLUNAS_NORM])

print("\n[NORMALIZAÇÃO] Min-Max aplicada às notas e médias")
print(f"  Faixa original   MEDIA_GERAL: {df['MEDIA_GERAL'].min():.1f} – {df['MEDIA_GERAL'].max():.1f}")
print(f"  Faixa normalizada MEDIA_GERAL: {df_norm['MEDIA_GERAL_NORM'].min():.3f} – {df_norm['MEDIA_GERAL_NORM'].max():.3f}")

In [ ]:
# 3.6 Visualização
CORES_NIVEL = {"Insuficiente": "#e15759", "Regular": "#f28e2b", "Bom": "#4e79a7", "Excelente": "#59a14f"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Passo 3 — Transformação: Variável-Alvo e Distribuição", fontsize=13, fontweight="bold")

ax = axes[0]
ax.hist(df["MEDIA_GERAL"], bins=60, color="#4e79a7", edgecolor="white", alpha=0.8)
for lim, rot in zip(LIMITES[1:-1], ROTULOS[:-1]):
    ax.axvline(lim, color="red", linestyle="--", linewidth=1.2)
    ax.text(lim + 5, ax.get_ylim()[1] * 0.95, rot, fontsize=8, color="red", rotation=90, va="top")
ax.set_title("Histograma da Média Geral")
ax.set_xlabel("Média Geral (pontos)")
ax.set_ylabel("Frequência")

ax = axes[1]
sizes  = [contagem.get(r, 0) for r in ROTULOS]
colors = [CORES_NIVEL[r] for r in ROTULOS]
wedges, texts, autotexts = ax.pie(sizes, labels=ROTULOS, colors=colors,
                                  autopct="%1.1f%%", startangle=140, pctdistance=0.75)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title("Distribuição de Nível de Desempenho")

plt.tight_layout()
plt.savefig("passo3_transformacao.png", dpi=120, bbox_inches="tight")
plt.show()   # [FIX] original usava plt.close() e o gráfico não aparecia

In [ ]:
# 3.7 Features finais para ML + salvar
FEATURES = (
    ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO",
     "MEDIA_OBJETIVAS", "DESVIO_NOTAS", "LOCALIZACAO_RURAL",
     "MELHOR_AREA_COD", "PIOR_AREA_COD"]
    + [c for c in df.columns if c.startswith("ESCOLA_")]
)
TARGET = "NIVEL_DESEMPENHO"

print(f"\n[FEATURES FINAIS] ({len(FEATURES)} atributos):")
for f in FEATURES:
    print(f"  • {f}")

df.to_csv("dados_transformados.csv", index=False, encoding="utf-8")
print(f"\n[OK] Arquivo salvo: dados_transformados.csv  ({len(df):,} registros)")

## Reflexão (preparação para o formulário)

1. Qual viés o filtro de presença completa introduz? Como documentá-lo?
2. Por que nota zero de candidato presente vira NaN, e por que remover em vez de imputar?
3. `MEDIA_GERAL` ficou fora de `FEATURES`, mas o alvo é função exata das 5 notas que estão dentro. Isso é vazamento de alvo? Que acurácia você esperaria de um modelo assim, e o que isso diz sobre a pergunta de pesquisa?
4. Por que One-Hot para `TIPO_ESCOLA` e Label Encoding para `MELHOR_AREA`?
5. Por que normalizar antes de um KNN?

_Suas respostas:_